# 04 Results tables and figures (Step 9)

Regenerates the tables and figures from the saved results (`dengue_recon.reporting`). The large prediction files (`preds_*.npz`) and the 300-municipality metrics file are not in the repository; figures that need them are kept from the pilot run and are re-created after running the `brazil` step (see `results/README.md`).

In [ ]:
# ---- Setup: works in Google Colab and locally (VS Code / Jupyter) ----
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/aminah9945/dengue-climate-reconstruction"
BRANCH = "main"          # use "official-run" if the pull request is not merged yet
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO = Path("/content/dengue-climate-reconstruction")
    if not REPO.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO}[all]"], check=True)
    # Optional: keep the downloaded data on Google Drive so it survives the session.
    USE_DRIVE = False
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        os.environ.setdefault("DATA_DIR", "/content/drive/MyDrive/dengue_recon_data")
else:
    # Local: this notebook lives in <repo>/notebooks/
    REPO = Path(os.environ.get("DENGUE_ROOT", Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())).resolve()

os.environ.setdefault("DENGUE_ROOT", str(REPO))
os.environ.setdefault("DATA_DIR", str(REPO / "data"))
os.environ.setdefault("RESULTS_DIR", str(REPO / "results"))
os.environ.setdefault("XLA_FLAGS", "--xla_force_host_platform_device_count=1")
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))      # makes the package importable without a kernel restart

from dengue_recon import config as C
C.ensure_dirs()
print("repo       :", C.ROOT)
print("DATA_DIR   :", C.DATA_DIR, "| processed data present:", (C.DATA_DIR / "dengue_weekly.parquet").exists())
print("RESULTS_DIR:", C.RESULTS_DIR)

In [ ]:
import json
import numpy as np, pandas as pd
from IPython.display import Image, Markdown, display
pd.set_option("display.width", 200, "display.max_columns", 30)


def show(path, width=900):
    p = Path(path)
    p = p if p.is_absolute() else C.FIG_DIR / p
    display(Image(str(p), width=width)) if p.exists() else print("missing figure:", p)


def have_data():
    ok = (C.DATA_DIR / "dengue_weekly.parquet").exists()
    if not ok:
        print("Processed data not found in DATA_DIR - run notebook 00 first (or set DATA_DIR).")
    return ok

In [ ]:
REGENERATE = False   # True rewrites results/ tables from the saved metrics (about 15 s)
if REGENERATE:
    from dengue_recon.reporting import rain_report, report
    report.main(); rain_report.main()
pd.read_csv(C.BRAZIL_DIR / "table_abs_means_ci.csv").round(4)

## Reproduction of the base paper

In [ ]:
show("fig_repro_lim.png")

## Baselines vs climate model: JSD and peak-week error with 95 % CIs

In [ ]:
show("fig_baseline_vs_climate_weekly_60.png"); show("fig_baseline_vs_climate_weekly_300.png")

## JSD distribution per model (weekly)

In [ ]:
show("fig_jsd_by_model.png")

## Gain by region and by year (300 municipalities)

In [ ]:
show("fig_gain_by_region_year_weekly_300.png", 1000)

## Example reconstructions (Belo Horizonte 2016 and 2019; largest M1 gain and loss)

In [ ]:
show("fig_example_bh.png", 1000)

## 300-municipality paired differences

In [ ]:
p = pd.read_csv(C.BRAZIL_DIR / "table_paired_weekly_300.csv")
p[(p.metric == "jsd")][["subset", "model_a", "model_b", "n", "mean_diff", "rel_change", "ci_muni_lo", "ci_muni_hi", "ci_year_lo", "ci_year_hi", "n_years_a_better"]].round(5)